In [1]:
from samap.mapping import SAMAP
from samap.analysis import get_mapping_scores
from samap.utils import save_samap, load_samap
from samalg import SAM

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import networkx as nx
import anndata as ad

from collections import defaultdict
import itertools, os, gc, pickle

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
LEVEL     = 'ss_subclass_v4_nounlabeled_nn'      # input cell-type label
OUT_LEVEL = 'ss_subclass_nounlabeled_nmm_v4_nn'  # consensus column written back
THRESH    = 0.2                                # SAMap score floor
DATE      = '07262026'

SAM_DIR   = 'Active_SAM_joined/'
SAMAP_DIR = 'Active_SAMap_Joined/active_samap/Non-mammal/'
BLOCKS    = f'nonmam_samap_blocks_{DATE}.pkl'    # cached mapping-table blocks

H5AD = {
    'cj': SAM_DIR + 'SAM_CJ_joined_v2_cleaned_03122025.h5ad',
    'ac': SAM_DIR + 'SAM_AC_ncbi_soupx_cleaned_03122025.h5ad',
    'xt': SAM_DIR + 'SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad',
    'dr': SAM_DIR + 'SAM_DR_ncbi_joined_cleaned_07172026.h5ad',
}

PAIRS = {
    ('cj', 'ac'): SAMAP_DIR + 'sm_cj_ac_ncbi_cleaned_03262025.pkl',
    ('cj', 'xt'): SAMAP_DIR + 'sm_cj_xt_cleaned_03262025.pkl',
    ('cj', 'dr'): SAMAP_DIR + 'sm_cj_v2_cleaned_dr_cleaned_07172026.pkl',
    ('ac', 'xt'): SAMAP_DIR + 'sm_ac_ncbi_xt_cleaned_03262025.pkl',
    ('ac', 'dr'): SAMAP_DIR + 'sm_ac_cleaned_dr_cleaned_07172026.pkl',
    ('xt', 'dr'): SAMAP_DIR + 'sm_xt_cleaned_dr_cleaned_07172026.pkl',
}

SPECIES      = ['cj', 'ac', 'xt', 'dr']   # also the prefix order used in group names
PREFIX_COLOR = {'cj': 'red', 'ac': 'blue', 'xt': 'green', 'dr': 'purple'}

print('cached blocks:', 'found' if os.path.exists(BLOCKS) else 'absent (section 2 will rebuild)')
for f in list(H5AD.values()) + list(PAIRS.values()):
    print(('OK   ' if os.path.exists(f) else 'MISS '), f)

cached blocks: found
OK    Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad
OK    Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad
OK    Active_SAM_joined/SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad
OK    Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad
OK    Active_SAMap_Joined/active_samap/Non-mammal/sm_cj_ac_ncbi_cleaned_03262025.pkl
OK    Active_SAMap_Joined/active_samap/Non-mammal/sm_cj_xt_cleaned_03262025.pkl
OK    Active_SAMap_Joined/active_samap/Non-mammal/sm_cj_v2_cleaned_dr_cleaned_07172026.pkl
OK    Active_SAMap_Joined/active_samap/Non-mammal/sm_ac_ncbi_xt_cleaned_03262025.pkl
OK    Active_SAMap_Joined/active_samap/Non-mammal/sm_ac_cleaned_dr_cleaned_07172026.pkl
OK    Active_SAMap_Joined/active_samap/Non-mammal/sm_xt_cleaned_dr_cleaned_07172026.pkl


## 1. Labels and node set

A label becomes a node if it is prefixed with its own species code, is not a mouse-merged `xx_m…`
type, and does not end in `NN`.

In [3]:
labels = {}   # species -> Series indexed by barcode

for s in SPECIES:
    a = ad.read_h5ad(H5AD[s], backed='r')
    labels[s] = a.obs[LEVEL].astype(str)
    print(f'{s}: {a.n_obs} cells, {labels[s].nunique()} distinct labels')
    del a

cj: 74758 cells, 139 distinct labels
ac: 47900 cells, 130 distinct labels
xt: 42221 cells, 123 distinct labels
dr: 61430 cells, 62 distinct labels


In [4]:
def is_neuronal(item):
    """Non-neuronal subclasses are suffixed NN, e.g. '326 OPC NN'."""
    return not item.endswith('NN')


def is_species_cluster(item, prefix):
    """cj_1 -> True; cj_m066_m067 -> False; '128 VMH Fezf1 Glut' / 'Unlabeled' -> False"""
    return len(item) > 3 and item[:2] == prefix and item[2] == '_' and item[3] != 'm'


node_species = {}
for s in SPECIES:
    keep = [i for i in labels[s].unique() if is_species_cluster(i, s)]
    node_species.update({i: s for i in keep})
    print(f'{s}: {len(keep)} neuronal nodes')
    print({i: s for i in keep})

cell_counts = {n: int(labels[node_species[n]].value_counts().get(n, 0)) for n in node_species}
print('\ntotal nodes:', len(node_species))

cj: 35 neuronal nodes
{'cj_3': 'cj', 'cj_8': 'cj', 'cj_4': 'cj', 'cj_5': 'cj', 'cj_18': 'cj', 'cj_7': 'cj', 'cj_21': 'cj', 'cj_25': 'cj', 'cj_6': 'cj', 'cj_11': 'cj', 'cj_19': 'cj', 'cj_9': 'cj', 'cj_13': 'cj', 'cj_14': 'cj', 'cj_22': 'cj', 'cj_20': 'cj', 'cj_2': 'cj', 'cj_27': 'cj', 'cj_1': 'cj', 'cj_33': 'cj', 'cj_16': 'cj', 'cj_26': 'cj', 'cj_34': 'cj', 'cj_35': 'cj', 'cj_32': 'cj', 'cj_28': 'cj', 'cj_30': 'cj', 'cj_10': 'cj', 'cj_24': 'cj', 'cj_12': 'cj', 'cj_17': 'cj', 'cj_29': 'cj', 'cj_36': 'cj', 'cj_31': 'cj', 'cj_37': 'cj'}
ac: 31 neuronal nodes
{'ac_20': 'ac', 'ac_0': 'ac', 'ac_4': 'ac', 'ac_13': 'ac', 'ac_3': 'ac', 'ac_11': 'ac', 'ac_19': 'ac', 'ac_1': 'ac', 'ac_7': 'ac', 'ac_17': 'ac', 'ac_28': 'ac', 'ac_32': 'ac', 'ac_8': 'ac', 'ac_22': 'ac', 'ac_34': 'ac', 'ac_30': 'ac', 'ac_2': 'ac', 'ac_16': 'ac', 'ac_14': 'ac', 'ac_26': 'ac', 'ac_6': 'ac', 'ac_35': 'ac', 'ac_10': 'ac', 'ac_9': 'ac', 'ac_27': 'ac', 'ac_18': 'ac', 'ac_25': 'ac', 'ac_31': 'ac', 'ac_24': 'ac', 'ac_29': 'ac

## 2. Mapping-table blocks

One block per pair: rows = species `b` types, columns = species `a` types, restricted to graph
nodes. Cached to `BLOCKS` — the SAMap pickles are only loaded if the cache is missing.

In [7]:
def eligible(block):
    return block.loc[[i for i in block.index   if i[3:] in node_species],
                     [c for c in block.columns if c[3:] in node_species]]


def compute_blocks():
    """Load each SAMap object once and keep the node-restricted score block."""
    out = {}
    out_test = {}
    for (a, b), path in PAIRS.items():
        print(f'=== {a} vs {b} : {os.path.basename(path)}')
        sm = load_samap(path)
        for org in (a, b):
            if LEVEL not in sm.sams[org].adata.obs.columns:
                raise KeyError(f'{org}: {LEVEL} missing from {os.path.basename(path)}')

        D, MappingTable = get_mapping_scores(sm, {a: LEVEL, b: LEVEL})
        full = MappingTable.loc[[i for i in MappingTable.index   if i[:2] == b],
                                [i for i in MappingTable.columns if i[:2] == a]]
        out[(a, b)] = eligible(full)
        out_test[(a,b)] = full
        print(f'    {full.shape} -> eligible {out[(a, b)].shape}')

        del sm, D, MappingTable, full
        gc.collect()
    return(out, out_test)


if os.path.exists(BLOCKS):
    with open(BLOCKS, 'rb') as fh:
        cached = pickle.load(fh)
    blocks = cached['eligible'] if isinstance(cached, dict) and 'eligible' in cached else cached
    print(f'loaded {len(blocks)} blocks from {BLOCKS}')
else:
    blocks, full_blocks = compute_blocks()
    with open(BLOCKS, 'wb') as fh:
        pickle.dump({'eligible': blocks}, fh)
    with open('test_full_' + BLOCKS, 'wb') as fh:
        pickle.dump({'eligible': full_blocks}, fh)
    print(f'saved {BLOCKS}')

for k, v in sorted(blocks.items()):
    print(f'  {k}: {v.shape}')

loaded 6 blocks from nonmam_samap_blocks_07262026.pkl
  ('ac', 'dr'): (28, 31)
  ('ac', 'xt'): (40, 31)
  ('cj', 'ac'): (31, 35)
  ('cj', 'dr'): (28, 35)
  ('cj', 'xt'): (40, 35)
  ('xt', 'dr'): (28, 40)


In [9]:
blocks[('cj', 'ac')]

,cj_cj_1,cj_cj_10,cj_cj_11,cj_cj_12,cj_cj_13,cj_cj_14,cj_cj_16,cj_cj_17,cj_cj_18,cj_cj_19,...,cj_cj_34,cj_cj_35,cj_cj_36,cj_cj_37,cj_cj_4,cj_cj_5,cj_cj_6,cj_cj_7,cj_cj_8,cj_cj_9
ac_ac_0,0.080706,0.000000,0.253084,0.001446,0.000306,0.000696,0.000000,0.001121,0.000632,0.021431,...,0.000000,0.000000,0.0,0.0,0.001221,0.013197,0.118787,0.007269,0.005461,0.003117
ac_ac_1,0.535631,0.000000,0.003975,0.025504,0.000000,0.000287,0.000021,0.053725,0.003123,0.006691,...,0.000000,0.000000,0.0,0.0,0.001251,0.000596,0.007677,0.001179,0.001692,0.003105
ac_ac_10,0.002739,0.000000,0.000000,0.000248,0.000000,0.000000,0.000000,0.000814,0.000000,0.000183,...,0.000000,0.000000,0.0,0.0,0.000000,0.000148,0.000235,0.000000,0.000303,0.000000
ac_ac_11,0.000102,0.000000,0.004592,0.000000,0.138985,0.006390,0.000000,0.000000,0.004507,0.009967,...,0.000000,0.000000,0.0,0.0,0.003724,0.000000,0.003101,0.000032,0.000152,0.001136
ac_ac_13,0.004422,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000151,0.000000,...,0.000000,0.004552,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ac_ac_14,0.441484,0.000000,0.000000,0.009100,0.000000,0.000296,0.000000,0.394345,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000102,0.000000,0.000000,0.000045,0.000000,0.000000
ac_ac_16,0.000000,0.000000,0.000000,0.000000,0.000000,0.000173,0.000358,0.000000,0.000000,0.003264,...,0.000000,0.000000,0.0,0.0,0.000022,0.000000,0.000000,0.000000,0.000000,0.000000
ac_ac_17,0.001386,0.000000,0.000000,0.000000,0.000000,0.003442,0.000000,0.000000,0.000072,0.000000,...,0.000000,0.000000,0.0,0.0,0.000116,0.050287,0.001468,0.645494,0.004698,0.000000
ac_ac_18,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000301,0.000000,0.000000,0.000086,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ac_ac_19,0.010931,0.000000,0.000047,0.000911,0.000000,0.000111,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000044


## 3. Best hits among eligible partners

`idxmax` over the node-only block, so every recorded partner is one that can form an edge.

In [10]:
def best_hits_from(block):
    out = {}
    for col in block.columns:
        item = col[3:]
        if block[col].isna().all() or block[col].max() == 0:
            out[item] = [None, 0.0]
            continue
        best = block[col].idxmax()
        out[item] = [best[3:], float(block.loc[best, col])]
    return out


mapping_frames = {}
for (a, b), blk in blocks.items():
    for (src, dst), d in (((a, b), best_hits_from(blk)), ((b, a), best_hits_from(blk.T))):
        df = pd.DataFrame.from_dict(d, orient='index',
                                    columns=['SAMAP match', 'SAMAP score'])
        df.index.name = 'source'
        fn = f'{src}_{dst}_nonmam_mapping_SAMap_{DATE}.csv'
        df.to_csv(fn)
        mapping_frames[(src, dst)] = df
        print(f'{fn}  ({len(df)} types, '
              f'{int((df["SAMAP score"] >= THRESH).sum())} at or above {THRESH})')

cj_ac_nonmam_mapping_SAMap_07262026.csv  (35 types, 12 at or above 0.2)
ac_cj_nonmam_mapping_SAMap_07262026.csv  (31 types, 13 at or above 0.2)
cj_xt_nonmam_mapping_SAMap_07262026.csv  (35 types, 14 at or above 0.2)
xt_cj_nonmam_mapping_SAMap_07262026.csv  (40 types, 12 at or above 0.2)
cj_dr_nonmam_mapping_SAMap_07262026.csv  (35 types, 8 at or above 0.2)
dr_cj_nonmam_mapping_SAMap_07262026.csv  (28 types, 4 at or above 0.2)
ac_xt_nonmam_mapping_SAMap_07262026.csv  (31 types, 14 at or above 0.2)
xt_ac_nonmam_mapping_SAMap_07262026.csv  (40 types, 15 at or above 0.2)
ac_dr_nonmam_mapping_SAMap_07262026.csv  (31 types, 9 at or above 0.2)
dr_ac_nonmam_mapping_SAMap_07262026.csv  (28 types, 6 at or above 0.2)
xt_dr_nonmam_mapping_SAMap_07262026.csv  (40 types, 12 at or above 0.2)
dr_xt_nonmam_mapping_SAMap_07262026.csv  (28 types, 10 at or above 0.2)


## 4. Edges: reciprocal best hit ≥ THRESH

In [11]:
G = nx.Graph()
G.add_nodes_from(node_species)

for (a, b) in PAIRS:
    for (src, dst) in ((a, b), (b, a)):
        fwd, rev = mapping_frames[(src, dst)], mapping_frames[(dst, src)]
        for u, row in fwd.iterrows():
            v = row['SAMAP match']
            score = float(row['SAMAP score'])
            if v in rev.index:
                if rev.loc[v, 'SAMAP match'] == u and score > THRESH:
                    G.add_edge(u, v, weight=score)

print(f'edges: {G.number_of_edges()}')

edges: 50


In [12]:
THRESH

0.2

In [13]:
pd.DataFrame([{'A': u, 'B': v, 'score': round(d['weight'], 3)}
               for u, v, d in G.edges(data=True)]).sort_values('score', ascending=False)

,A,B,score
35,ac_34,xt_32,0.913
22,cj_12,ac_27,0.847
36,ac_30,xt_38,0.798
42,xt_15,dr_27,0.660
24,cj_17,ac_9,0.647
2,cj_7,ac_17,0.645
47,xt_37,dr_7,0.603
13,cj_16,ac_6,0.592
4,cj_6,xt_21,0.572
23,cj_12,dr_0,0.572


## 5. Consensus groups

Connected components become homology groups, named for the species present in `cj, ac, xt, dr`
order plus a counter (`cj_ac_xt_1`). Types in no group keep their original label.

In [14]:
def name_groups(g):
    comps = [c for c in nx.connected_components(g) if len(c) > 1]
    counters = defaultdict(int)
    res, members = {}, {}
    for c in sorted(comps, key=len, reverse=True):
        combo = '_'.join(sorted({node_species[n] for n in c}, key=SPECIES.index))
        counters[combo] += 1
        name = f'{combo}_{counters[combo]}'
        members[name] = sorted(c)
        for n in c:
            res[n] = name
    return res, members


result, members = name_groups(G)
print(f'{len(members)} groups covering {len(result)} of {len(node_species)} types\n')

for name, mem in sorted(members.items(), key=lambda kv: -len(kv[1])):
    counts = {s: sum(1 for n in mem if node_species[n] == s) for s in SPECIES}
    dup = ', '.join(f'{s}x{counts[s]}' for s in SPECIES if counts[s] > 1)
    print(f'=== {name}   {len(mem)} types, {sum(cell_counts[n] for n in mem)} cells'
          + (f'   [DUPES: {dup}]' if dup else ''))
    for n in sorted(mem):
        nbrs = sorted(G[n].items(), key=lambda kv: -kv[1]['weight'])
        print(f"    {n:<8} {cell_counts[n]:>6} cells   -> "
              + ', '.join(f"{m}={d['weight']:.2f}" for m, d in nbrs))
    print()

21 groups covering 66 of 134 types

=== cj_ac_xt_dr_1   7 types, 9160 cells   [DUPES: cjx2, acx3]
    ac_1       1481 cells   -> xt_1=0.57, cj_1=0.54
    ac_23       103 cells   -> dr_0=0.38
    ac_27        75 cells   -> cj_12=0.85
    cj_1       1025 cells   -> ac_1=0.54, xt_1=0.48
    cj_12       321 cells   -> ac_27=0.85, dr_0=0.57
    dr_0       5329 cells   -> cj_12=0.57, ac_23=0.38, xt_1=0.30
    xt_1        826 cells   -> ac_1=0.57, cj_1=0.48, dr_0=0.30

=== cj_ac_xt_dr_2   7 types, 3909 cells   [DUPES: cjx2, xtx3]
    ac_6        561 cells   -> cj_16=0.59, xt_4=0.44, dr_7=0.29
    cj_16       248 cells   -> ac_6=0.59, dr_7=0.55, xt_26=0.37
    cj_30        71 cells   -> xt_4=0.29
    dr_7       2216 cells   -> xt_37=0.60, cj_16=0.55, ac_6=0.29
    xt_26       146 cells   -> cj_16=0.37
    xt_37        62 cells   -> dr_7=0.60
    xt_4        605 cells   -> ac_6=0.44, cj_30=0.29

=== cj_ac_xt_1   6 types, 3919 cells   [DUPES: cjx2, acx2, xtx2]
    ac_0       1851 cells   -> cj_3

In [15]:
summary = pd.DataFrame([
    {'group': name,
     'types': len(mem),
     'cells': sum(cell_counts[n] for n in mem),
     'species': len({node_species[n] for n in mem}),
     'dupes': ', '.join(f'{s}x{c}' for s, c in
                        ((s, sum(1 for n in mem if node_species[n] == s)) for s in SPECIES)
                        if c > 1),
     'min score': round(min(d['weight'] for _, _, d in G.subgraph(mem).edges(data=True)), 3),
     'members': ' '.join(mem)}
    for name, mem in members.items()]).sort_values(['species', 'types'], ascending=False)

pd.set_option('display.max_colwidth', 200)
summary.to_csv(f'nonmam_consensus_groups_SAMap_{DATE}.csv', index=False)
pd.Series(result, name='group').sort_index().to_frame().to_csv(
    f'nonmam_consensus_assignment_SAMap_{DATE}.csv')
summary.set_index('group')

,types,cells,species,dupes,min score,members
group,,,,,,
cj_ac_xt_dr_1,7,9160,4,"cjx2, acx3",0.298,ac_1 ac_23 ac_27 cj_1 cj_12 dr_0 xt_1
cj_ac_xt_dr_2,7,3909,4,"cjx2, xtx3",0.291,ac_6 cj_16 cj_30 dr_7 xt_26 xt_37 xt_4
cj_ac_xt_dr_3,4,1794,4,,0.292,ac_13 cj_20 dr_13 xt_35
cj_ac_xt_dr_4,4,6828,4,,0.296,ac_3 cj_2 dr_1 xt_0
cj_ac_xt_dr_5,4,3813,4,,0.248,ac_8 cj_10 dr_5 xt_27
cj_ac_xt_1,6,3919,3,"cjx2, acx2, xtx2",0.252,ac_0 ac_22 cj_11 cj_3 xt_10 xt_5
cj_xt_dr_1,4,3284,3,drx2,0.213,cj_26 dr_27 dr_6 xt_15
cj_xt_dr_2,3,3500,3,,0.204,cj_25 dr_4 xt_6
ac_xt_dr_1,3,1066,3,,0.352,ac_20 dr_18 xt_30


In [16]:
def apply_labels(fn, s, dry_run=True):
    sam = SAM()
    sam.load_data(fn)

    src = sam.adata.obs[LEVEL].astype(str)
    fin = src.map(lambda x: result.get(x, x))

    print(f'{s}: {int((fin != src).sum())}/{len(src)} cells relabelled, '
          f'{src.nunique()} -> {fin.nunique()} distinct labels')
    print('    new labels:', sorted(set(fin) - set(src)))

    if dry_run:
        print('    dry run — nothing written')
        return sam

    sam.adata.obs[OUT_LEVEL] = pd.Categorical(fin.values)
    sam.save_anndata(fn)
    print(f'    saved {OUT_LEVEL} into {fn}')
    return sam

In [17]:
# Flip to dry_run=False to commit. Overwrites the four h5ads.
for s in SPECIES:
    apply_labels(H5AD[s], s, dry_run=False)
    gc.collect()

cj: 5778/74758 cells relabelled, 139 -> 136 distinct labels
    new labels: ['cj_ac_1', 'cj_ac_2', 'cj_ac_3', 'cj_ac_4', 'cj_ac_xt_1', 'cj_ac_xt_dr_1', 'cj_ac_xt_dr_2', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4', 'cj_ac_xt_dr_5', 'cj_xt_1', 'cj_xt_dr_1', 'cj_xt_dr_2']
    saved ss_subclass_nounlabeled_nmm_v4_nn into Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad
ac: 8746/47900 cells relabelled, 130 -> 127 distinct labels
    new labels: ['ac_dr_1', 'ac_dr_2', 'ac_xt_1', 'ac_xt_2', 'ac_xt_3', 'ac_xt_4', 'ac_xt_dr_1', 'cj_ac_1', 'cj_ac_2', 'cj_ac_3', 'cj_ac_4', 'cj_ac_xt_1', 'cj_ac_xt_dr_1', 'cj_ac_xt_dr_2', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4', 'cj_ac_xt_dr_5']
    saved ss_subclass_nounlabeled_nmm_v4_nn into Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad
xt: 6084/42221 cells relabelled, 123 -> 120 distinct labels
    new labels: ['ac_xt_1', 'ac_xt_2', 'ac_xt_3', 'ac_xt_4', 'ac_xt_dr_1', 'cj_ac_xt_1', 'cj_ac_xt_dr_1', 'cj_ac_xt_dr_2', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4', 'cj_ac_xt_d